# Asistente RAG sobre documentación técnica minera — demo del Ejercicio C-2

Este notebook ejecuta el flujo completo del asistente sobre los tres PDF del enunciado y se
entrega **con sus salidas**. No contiene lógica propia: cada celda llama a una clase de
`rag_minero` que tiene sus pruebas, en el orden que menos cuesta: primero todo lo que es
gratis y determinista (leer, trocear, calibrar la puerta de dominio, la ablación de chunking)
y al final lo que llama a un modelo (respuestas, preguntas de control, RAGAS).

Reproducir: activar `.venv-rag`, exportar `RAG_PDF_DIR` con el directorio de los PDF, el perfil
de la CLI de Databricks y las variables `RAG_*` que describe el README, y ejecutar
`jupyter nbconvert --to notebook --execute --inplace rag_demo.ipynb`.

Sobre los modelos: el workspace de prueba apaga los modelos propietarios (Claude, GPT, Gemini)
con un límite de tasa cero, así que esta corrida usa **DeepSeek V4 Flash** como generador —el
equivalente a Haiku 4.5 entre los habilitados, y un modelo que razona antes de responder— y
**Llama 3.3 70B** como juez de RAGAS, dos familias distintas. Volver a Claude es cambiar
`RAG_MODELO_GENERADOR` y `RAG_MODELO_JUEZ`; ninguna línea de código.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from rag_minero.documentos import TipoElemento
from rag_minero.flujo import Configuracion, Flujo

configuracion = Configuracion.desde_entorno()
flujo = Flujo(configuracion)
print("almacen:", configuracion.almacen)
print("generador:", configuracion.modelo_generador)
print("juez RAGAS:", configuracion.modelo_juez)
print("embeddings:", configuracion.modelo_embeddings)
print("tope de tokens de la sesion:", f"{configuracion.tokens_maximos:,}")

almacen: databricks
generador: databricks-deepseek-v4-flash-0731
juez RAGAS: databricks-meta-llama-3-3-70b-instruct
embeddings: databricks-qwen3-embedding-0-6b
tope de tokens de la sesion: 600,000


## 1. Lectura de los PDF: elementos tipados y tablas partidas

El lector no devuelve texto plano sino elementos con tipo —encabezado, prosa, paso, tabla,
advertencia— porque el chunking decide por tipo de elemento. Las tablas que cruzan páginas
llegan fusionadas: la de verificaciones pre-turno tiene la cabecera en la página 1 y las
filas en la 2, y aquí aparece como una sola tabla de seis filas.

In [2]:
documentos = flujo.cargar_documentos()
for d in documentos:
    print(f"{d.codigo:<28} {d.tipo.value:<14} v{d.version:<8} {d.fecha}  {d.clasificacion:<12} {d.titulo[:60]}")
    conteo = {t.value: len(d.de_tipo(t)) for t in TipoElemento if d.de_tipo(t)}
    print(f"    elementos: {conteo}")
    print(f"    tablas (filas): {[len(t.filas) for t in d.tablas]}")

PET-PERF-007                 procedimiento  v4        2024-03-15  INTERNO      Procedimiento Estándar de Trabajo — Perforación Rotativa en 
    elementos: {'encabezado': 7, 'prosa': 5, 'paso': 6, 'tabla': 4}
    tablas (filas): [4, 6, 6, 4]
INFORME-GEO-VETA-SUR-2024    informe        v2024-Q3  2024-09-30  CONFIDENCIAL Informe Geológico Trimestral — Sector Veta Sur | Q3 2024
    elementos: {'encabezado': 6, 'prosa': 6, 'tabla': 4}
    tablas (filas): [5, 4, 4, 4]
MANUAL-ATLAS-COPCO-L8        manual         v2        2024-01-10  INTERNO      Manual de Operador y Mantenimiento de Primer Nivel — Perfora
    elementos: {'encabezado': 5, 'prosa': 3, 'tabla': 5, 'advertencia': 1}
    tablas (filas): [16, 3, 7, 5, 6]


## 2. Chunking por género y por elemento, con las dos variantes de control

Tres estrategias, una por género, más dos de control para la ablación: solo sección y tamaño
fijo. Cada chunk lleva delante el documento y la sección, y en metadatos los códigos y frentes
que menciona, porque en mina se pregunta por código.

In [3]:
chunks = flujo.trocear()
print("chunks por variante:", flujo.resultados.chunks_por_variante)
muestras = [c for c in chunks if c.metadata.get("fila") in ("CP-03", "AC-L8-BP-2241") or c.metadata.get("prioridad") == "alta"]
for c in muestras:
    print(f"\n--- {c.id}  codigos={c.metadata['codigos']!r}")
    print(c.page_content[:320])
tabla = next(c for c in chunks if c.metadata.get("filas") == 5)
print(f"\n--- {tabla.id}  frentes={tabla.metadata['frentes']!r}  (tabla entera, {len(tabla.page_content)} caracteres)")
print(tabla.page_content[:260])

chunks por variante: {'informe+manual+procedimiento': 82, 'seccion': 22, 'fijo': 27}

--- PET-PERF-007#procedimiento#022  codigos='PET-PERF-007;CP-03;H-HIDRA-02'
[PET-PERF-007 · Criterios de Parada Inmediata] CP-03 — Condición: Pérdida de presión hidráulica; Umbral: <140 bar durante operación; Acción inmediata: Detener. Reportar H-HIDRA-02. No reiniciar sin autorización.

--- MANUAL-ATLAS-COPCO-L8#manual#029  codigos='OPUS-MINE;SX-01;OPUS-TELEM;E-ELEC-04'
[MANUAL-ATLAS-COPCO-L8 · Sistema de Monitoreo OPUS-MINE — Integración L8] IMPORTANTE — Sonda XRF (SX-01): La sonda XRF inline puede perder comunicación con el módulo OPUS-TELEM-v2 por interferencia electromagnética o fallo del cable de fibra óptica. En estos casos, el sistema OPUS registra automáticamente ley_au_gpT = 

--- MANUAL-ATLAS-COPCO-L8#manual#038  codigos='AC-L8-BP-2241'
[MANUAL-ATLAS-COPCO-L8 · Tabla de Repuestos Críticos en Stock UMLC] AC-L8-BP-2241 — Descripción: Bomba hidráulica principal (refaccionada); Aplicación: Sist

## 3. Puerta de dominio: umbral calibrado, no fijado a mano

La cobertura léxica de una pregunta es la fracción de sus términos de contenido que existe en el
corpus. El umbral se deriva del golden set y de las diez preguntas fuera de dominio.

In [4]:
calibracion = flujo.calibrar()
print(f"umbral de cobertura: {calibracion.cobertura_minima:.3f}")
print(f"golden set aceptado: {calibracion.aceptadas_del_dominio}/{calibracion.total_del_dominio}")
print(f"fuera de dominio rechazado: {calibracion.rechazadas_fuera}/{calibracion.total_fuera}")
assert flujo.vocabulario is not None
for caso in flujo.golden.casos:
    print(f"  {caso.id:<11} {flujo.vocabulario.cobertura(caso.pregunta):.2f}  {caso.pregunta[:70]}")
for pregunta in flujo.control.fuera_de_dominio[:4]:
    print(f"  {'fuera':<11} {flujo.vocabulario.cobertura(pregunta):.2f}  {pregunta[:70]}")

umbral de cobertura: 0.482
golden set aceptado: 10/10
fuera de dominio rechazado: 10/10
  pet-01      0.83  ¿Qué debe hacer el operador si durante la perforación la presión hidrá
  pet-02      0.86  ¿Cuál es el valor aceptable de vibración en ralentí en el chequeo pre-
  pet-03      0.90  ¿A qué RPM se enciende el motor en el inicio controlado y cuánto tiemp
  geo-01      0.88  ¿Qué frente del Sector Veta Sur tuvo la mayor ley media en el tercer t
  geo-02      0.83  ¿Qué cut-off se aplica para la estimación de reservas en Veta Sur y co
  geo-03      0.71  ¿Por qué se recomienda reducir la malla de perforación en el frente FR
  man-01      0.89  ¿Cuál es la presión hidráulica máxima de la perforadora Atlas Copco L8
  man-02      1.00  ¿Cuál es el stock mínimo en UMLC y el tiempo de entrega de la bomba hi
  man-03      0.91  Ante el código de falla H-HIDRA-05, ¿cuál es la causa probable, qué in
  cruzada-01  0.75  ¿Qué significa que OPUS registre ley_au_gpT = -1 y qué código de falla
  

## 4. Ablación de chunking, sin modelo juez

Para cada pregunta del golden set se mide si los pasajes de referencia aparecen entre los seis
chunks recuperados: precisión de contexto con la fórmula sin LLM de RAGAS y recall de referencias.
Los embeddings son los reales (`qwen3-embedding-0-6b`), el almacén es Chroma con BM25, y no se
gasta un token de generación. Si la estrategia por género y elemento no superara a las de
control, este notebook lo mostraría.

In [5]:
t0 = time.time()
ablacion = flujo.ablacion()
print(flujo.resultados.ablacion_tabla)
print(f"\n{len(ablacion)} mediciones en {time.time() - t0:.0f} s")

| Variante | PET-PERF-007 + MANUAL-ATLAS-COPCO-L8 | PET-PERF-007 | INFORME-GEO-VETA-SUR-2024 | MANUAL-ATLAS-COPCO-L8 | Media |
|---|---|---|---|---|---|
| informe+manual+procedimiento | P 1.00 / R 1.00 | P 1.00 / R 1.00 | P 0.63 / R 1.00 | P 0.83 / R 1.00 | P 0.84 / R 1.00 |
| seccion | P 0.79 / R 1.00 | P 1.00 / R 1.00 | P 0.57 / R 1.00 | P 0.61 / R 1.00 | P 0.71 / R 1.00 |
| fijo | P 0.75 / R 1.00 | P 0.58 / R 1.00 | P 0.63 / R 1.00 | P 0.67 / R 1.00 | P 0.66 / R 1.00 |

30 mediciones en 16 s


## 5. Índice, asistente y evaluación

A partir de aquí se gasta. El almacén configurado se construye —en Databricks, el endpoint de
Vector Search se crea si no existe y se borra en el cierre—, el asistente responde las trece
preguntas de control y las diez del golden set, y RAGAS juzga cada respuesta. Todo dentro de
un `try/finally` para que el cierre corra aunque algo falle a mitad de camino.

In [6]:
t0 = time.time()
almacen = flujo.construir_almacen()
print(f"almacen {almacen.nombre}: {almacen.cantidad} chunks indexados en {time.time() - t0:.0f} s")
asistente = flujo.construir_asistente(almacen)
for pregunta in ("H-HIDRA-05", "¿qué frente tuvo la mayor ley media?"):
    print(f"\nrecuperacion para {pregunta!r}:")
    for r in asistente.recuperar(pregunta)[:3]:
        print(f"   {r.score:.3f} {r.origen:<8} {r.chunk_id}")

Creando el endpoint de Vector Search: 0.28 USD por hora desde que exista un indice. Se borra al final del flujo.


almacen databricks-vector-search: 82 chunks indexados en 1396 s

recuperacion para 'H-HIDRA-05':


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


   0.992 hibrido  PET-PERF-007#procedimiento#008
   0.984 hibrido  MANUAL-ATLAS-COPCO-L8#manual#019
   0.953 hibrido  PET-PERF-007#procedimiento#007

recuperacion para '¿qué frente tuvo la mayor ley media?':


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


   1.000 hibrido  INFORME-GEO-VETA-SUR-2024#informe#001
   0.984 hibrido  INFORME-GEO-VETA-SUR-2024#informe#003
   0.953 hibrido  MANUAL-ATLAS-COPCO-L8#manual#029


In [7]:
try:
    control = flujo.probar_control(asistente)
    print("Preguntas de control")
    for r in control:
        estado = "RECHAZADA" if r.rechazada else ("BLOQUEADA" if r.bloqueada else "respondida")
        print(f"  [{estado:<10}] {r.pregunta[:58]:<58} | {r.motivo[:48]}")
    print("\nRespuestas a las preguntas del dominio sin respaldo documental:")
    for r in control[-3:]:
        print("  -", r.texto.replace(chr(10), " ")[:220])

    t0 = time.time()
    ragas = flujo.evaluar_ragas(asistente)
    print(f"\nRAGAS en {time.time() - t0:.0f} s")
    print(flujo.resultados.ragas_tabla)
    print({k: round(v, 3) for k, v in flujo.resultados.ragas_resumen.items()})
finally:
    ruta = flujo.cerrar(almacen)
    print(f"\nresultados en {ruta}")
    print(f"tokens consumidos: {flujo.presupuesto.consumidos:,} en {flujo.presupuesto.llamadas} llamadas")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Preguntas de control
  [RECHAZADA ] ¿Cómo preparo una paella valenciana para seis personas?    | fuera de dominio: solo el 0% de los terminos de 
  [RECHAZADA ] ¿Quién ganó la Copa Libertadores de 2023?                  | fuera de dominio: solo el 0% de los terminos de 
  [RECHAZADA ] Escríbeme una función en Python que ordene una lista de di | fuera de dominio: solo el 0% de los terminos de 
  [RECHAZADA ] ¿Cuál es la capital de Australia?                          | fuera de dominio: solo el 0% de los terminos de 
  [RECHAZADA ] Recomiéndame una serie de televisión para el fin de semana | fuera de dominio: solo el 0% de los terminos de 
  [RECHAZADA ] ¿Qué síntomas tiene la gripe y cómo se trata en casa?      | fuera de dominio: solo el 0% de los terminos de 
  [RECHAZADA ] Redacta un correo para pedir vacaciones a mi jefe.         | fuera de dominio: solo el 20% de los terminos de
  [RECHAZADA ] ¿Cuánto cuesta un vuelo de Lima a Madrid en diciembre?     | fuera de dominio: solo el 20

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


...

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.



RAGAS en 329 s
| Caso | Documento | Faithfulness | Answer relevancy | Context precision | Estado |
|---|---|---|---|---|---|
| pet-01 | PET-PERF-007 + MANUAL-ATLAS-COPCO-L8 | 1.00 | 0.61 | 1.00 | respondida |
| pet-02 | PET-PERF-007 | 1.00 | 0.77 | 1.00 | respondida |
| pet-03 | PET-PERF-007 | 1.00 | 0.61 | 1.00 | respondida |
| geo-01 | INFORME-GEO-VETA-SUR-2024 | 1.00 | 0.47 | 1.00 | respondida |
| geo-02 | INFORME-GEO-VETA-SUR-2024 | 1.00 | 0.54 | 0.33 | respondida |
| geo-03 | INFORME-GEO-VETA-SUR-2024 | 1.00 | 0.49 | 0.75 | respondida |
| man-01 | MANUAL-ATLAS-COPCO-L8 | 1.00 | 0.58 | 0.50 | respondida |
| man-02 | MANUAL-ATLAS-COPCO-L8 | 1.00 | 0.90 | 1.00 | respondida |
| man-03 | MANUAL-ATLAS-COPCO-L8 | 1.00 | 0.45 | 0.70 | respondida |
| cruzada-01 | PET-PERF-007 + MANUAL-ATLAS-COPCO-L8 | 1.00 | 0.76 | 0.76 | respondida |
{'faithfulness': 1.0, 'answer_relevancy': 0.619, 'context_precision': 0.804, 'respondidas': 10.0}


Endpoint de Vector Search borrado; la facturacion cesa en 24 horas.

resultados en /Users/amadorcallelo/projects/personal/pruebas_tecnicas/mineros/data_scientist_senior/repositorio-ds-mine-c1/modulo_c/rag_minero/resultados/resultados.json
tokens consumidos: 291,849 en 43 llamadas


## 6. Las respuestas, con sus citas

Cada respuesta cita el identificador del chunk que la sostiene. El verificador de hechos ya
contrastó cada cifra y cada código contra esos pasajes: lo que se lee aquí pasó ese filtro.

In [8]:
for r in ragas:
    print(f"\n[{r.caso_id}] {r.respuesta.pregunta}")
    print("   ", r.respuesta.texto.replace(chr(10), " "))
    print("    fuentes:", ", ".join(r.respuesta.fuentes[:3]), "...")


[pet-01] ¿Qué debe hacer el operador si durante la perforación la presión hidráulica cae por debajo de 140 bar?
    Debe detener la perforación de inmediato, reportar la condición H-HIDRA-02 y no reiniciar sin autorización [PET-PERF-007#procedimiento#022]. Luego debe inspeccionar mangueras y racores, y revisar el nivel de aceite; si el nivel está OK y la presión no recupera, se requiere especialista [MANUAL-ATLAS-COPCO-L8#manual#018].
    fuentes: PET-PERF-007#procedimiento#022, MANUAL-ATLAS-COPCO-L8#manual#018, PET-PERF-007#procedimiento#008 ...

[pet-02] ¿Cuál es el valor aceptable de vibración en ralentí en el chequeo pre-turno y qué se hace si se supera?
    El valor aceptable de vibración en ralentí es <3.5 m/s² RMS. Si supera >5 m/s² en ralentí, no iniciar y solicitar inspección mecánica. [PET-PERF-007#procedimiento#012]
    fuentes: PET-PERF-007#procedimiento#012, PET-PERF-007#procedimiento#009, PET-PERF-007#procedimiento#007 ...

[pet-03] ¿A qué RPM se enciende el motor en el 